# COMPASS preprocessing

Stage 0-3 of the COMPASS survival pipeline: schema audit (profile_data only),
cohort compile, longitudinal lab preprocessing, prediction-input build, and
cohort diagnostics. Univariate/multivariate modeling live in
`02_univariate.ipynb` / `03_multivariate.ipynb` and only read the
`prediction_inputs_<arm>/` files this notebook writes. All stages use the
merged `profile_data` parquets.

In [ ]:
ARMS = ["adt"]

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

RUNS = cp.make_runs(ARMS)

## Stage 0 -- schema audit

Fails fast if a required column is absent or all-null in the merged sources.

In [ ]:
cp.audit_schema()

## Stage 1 -- compile COMPASS cohort data

In [ ]:
cp.compile_cohort(arms=ARMS)

## Stage 2 -- preprocess raw labs (per arm anchor)

Expensive: full raw lab standardization. The Parquet cache
(`consolidated_longitudinal_data_<arm>.parquet`) makes reruns cheap, but the
first pass may be slow.

In [ ]:
for run in RUNS:
    cp.preprocess_labs(run)

## Stage 3 -- build prediction inputs + cohort diagnostics

Set `REBUILD_PREDICTION_INPUTS = False` to skip rebuilding
`aggregated_landmark*.csv` / `pre_treatment_lab_long_landmark*.csv` when only
re-running diagnostics.

In [ ]:
REBUILD_PREDICTION_INPUTS = True

for run in RUNS:
    if REBUILD_PREDICTION_INPUTS:
        cp.build_prediction_inputs(run)
    else:
        print(f"[skip] prediction-input rebuild disabled for {run['label']}")
    cp.cohort_diagnostics(run)

## Stage 3b -- separate somatic + Gleason + biomarker PRS inputs

Builds `prediction_inputs_<arm>/somatic_gleason/` from the sample-level
somatic matrix published by `PROFILE_data_processing` and the Gleason timeline
published by `LLM_clinical_annotations`, plus the explicitly allowlisted PGS
columns from the germline matrix. This arm is built only at the baseline
landmark (+0 days), using information available by treatment initiation.

In [ ]:
for run in RUNS:
    cp.build_somatic_gleason_inputs(run)